# FER-CE Baseline: ResNet50 for Emotion Recognition

This notebook implements a baseline model using pre-trained ResNet50 with comprehensive evaluation metrics including training curves.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Path handling
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'notebooks':
    project_root = os.path.abspath('..')
else:
    project_root = os.path.abspath('.')

src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from dataset import RAFCEDataset, get_transforms

## 1. Parameters

In [ ]:
IMG_DIR = os.path.join(project_root, '../aligned')
LABEL_FILE = os.path.join(project_root, '../RAFCE_emolabel.txt')
PARTITION_FILE = os.path.join(project_root, '../RAFCE_partition.txt')
OUTPUT_DIR = os.path.join(project_root, 'outputs/baseline')

BATCH_SIZE = 32
EPOCHS = 20
LR = 0.001
NUM_CLASSES = 14
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")

## 2. Data Loading

In [ ]:
train_transform = get_transforms(augment=True)
val_transform = get_transforms(augment=False)

train_dataset = RAFCEDataset(IMG_DIR, LABEL_FILE, PARTITION_FILE, split=1, transform=train_transform)
test_dataset = RAFCEDataset(IMG_DIR, LABEL_FILE, PARTITION_FILE, split=2, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 3. Model Definition

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

## 4. Training

In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs):
    train_losses = []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")
    return train_losses

losses = train_model(model, train_loader, criterion, optimizer, EPOCHS)
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'resnet50_baseline.pth'))

## 5. Evaluation with Training Curves

In [ ]:
def evaluate(model, test_loader, train_losses):
    """
    Comprehensive evaluation with:
    - Training loss curves
    - Confusion matrix
    - Classification report
    """
    # 1. Plot Training Curves
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-', linewidth=2, marker='o', markersize=4)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training Loss Curve', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 2. Evaluate on test set
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # 3. Classification Report
    print("=" * 60)
    print("CLASSIFICATION REPORT")
    print("=" * 60)
    print(classification_report(all_preds, all_labels))
    
    # 4. Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 5. Final Accuracy
    accuracy = 100 * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
    print(f"\n{'='*60}")
    print(f"FINAL TEST ACCURACY: {accuracy:.2f}%")
    print(f"{'='*60}\n")

evaluate(model, test_loader, losses)